# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Dataset metadata (do not subscript, use attributes or .to_json())
metadata = dataset.metadata.to_json()
print(f"Dataset Name: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"Version: {metadata.get('version', 'Unknown')}")
print(f"Published: {metadata.get('datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, **record sets** represent logical tables. Each record set contains **fields** (columns), all uniquely addressed by their `@id`.

In [ ]:
# List all record sets by their @id and name attributes
print("Available Record Sets (by @id and name):")
recordsets = list(dataset.metadata.record_sets)
for rs in recordsets:
    print(f"- @id: {rs.id} | name: {getattr(rs, 'name', 'N/A')}")

# For each record set, list available fields and their @id
for rs in recordsets:
    print(f"\nRecord Set '@id': {rs.id} ({getattr(rs, 'name', 'N/A')})")
    print("Fields (columns):")
    for field in rs.fields:
        print(f"\t- @id: {field.id} | name: {getattr(field, 'name', 'N/A')} | dataType: {getattr(field, 'data_type', 'N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List the @ids of all record sets available (to help choose one for analysis)
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print("Record set @ids:", record_set_ids)

# Let's select the primary record set (use the first if not sure, or specify manually if known)
record_set_id = record_set_ids[0] if record_set_ids else None
if record_set_id is None:
    raise ValueError('No record set found in the dataset.')
print(f"Using Record Set: {record_set_id}\n")

# Extract all records from the selected record set
records = list(dataset.records(record_set=record_set_id))
df = pd.DataFrame(records)
print(f"Columns in DataFrame ({len(df.columns)} fields):", df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

We'll look for numeric columns to analyze, e.g., peptide counts, coverage percentages, or abundances.

In [ ]:
# Identify numeric columns (float/int) for analysis
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
print("Numeric columns available:", numeric_cols)

# For demonstration, pick the first numeric field (override as needed)
if numeric_cols:
    numeric_field = numeric_cols[0]
    print(f"Will analyze numeric field: {numeric_field}")
else:
    print("No numeric fields found.")
    numeric_field = None

if numeric_field:
    threshold = df[numeric_field].mean()  # For demonstration, filter above average
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} ({len(filtered_df)} records):")
    print(filtered_df.head())

    # Normalize the selected numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"].copy()].head())

    # If another categorical field exists, group by it
    cat_cols = df.select_dtypes(include=['object']).columns.tolist()
    group_field = None
    for col in cat_cols:
        if df[col].nunique() > 1 and df[col].nunique() < 20:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped data by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, we loaded the FAIR^2 mass spectrometry dataset using its Croissant schema and explored its tables (record sets) using the mlcroissant library.
* We listed all record sets and their fields using their `@id`s, and loaded data into pandas DataFrames for analysis.
* We identified and visualized numeric fields, applied normalization, and demonstrated basic group-by summaries for further applications.
* This approach ensures full traceability and reproducibility of FAIR datasets in your Python ML pipeline.

Feel free to further extend the analysis or visualization according to your research questions!